# 🔗 Blockchain Transaction Analytics — V2 (Fixed)
**Project:** Blockchain_Analysis  
**API Key Created:** 2026-03-26  
**Fix:** Updated to Etherscan API V2 endpoint  
**Fetches:** Wallet Transactions · Internal Transactions · ERC-20 Token Transfers · NFT Transfers · Smart Contract Event Logs

---
### ▶️ How to run
1. Click **Runtime → Run All**
2. Allow Google Drive access when prompted
3. All CSVs saved to `My Drive/etherscan_data/`

## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install requests pandas --quiet
print('✅ Libraries ready')

## ⚙️ Cell 2 — Configuration
**Everything is pre-filled. No changes needed unless you want a different wallet.**

In [ ]:
# ──────────────────────────────────────────────────────
# CONFIG — Pre-filled, no edits needed
# ──────────────────────────────────────────────────────

API_KEY       = "VSXII7UPIDDSH5YYKBNJNNVPY81QSGXQ3B"
WALLET        = "0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D"  # Uniswap V2 Router

# ✅ FIXED: Using Etherscan API V2
BASE_URL      = "https://api.etherscan.io/v2/api"
CHAIN_ID      = 1       # 1 = Ethereum mainnet

# Narrow block range to get faster results (recent ~1 million blocks = ~6 months)
START_BLOCK   = 19000000
END_BLOCK     = 99999999

PAGE_SIZE     = 10000
SLEEP_BETWEEN = 0.25

print('✅ Config set')
print(f'   Wallet  : {WALLET}')
print(f'   API URL : {BASE_URL}')
print(f'   Blocks  : {START_BLOCK} → {END_BLOCK}')

## ☁️ Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/etherscan_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Drive mounted. Files will be saved to: {OUTPUT_DIR}')

## 🛠️ Cell 4 — Core Fetch Functions

In [ ]:
import requests
import pandas as pd
import time

def fetch_all_pages(action, extra_params={}):
    """Paginates through all pages using Etherscan API V2."""
    all_records = []
    page = 1
    print(f'\n📡 Fetching [{action}] ...')

    while True:
        params = {
            'chainid':    CHAIN_ID,       # ✅ V2 requires chainid
            'module':     'account',
            'action':     action,
            'address':    WALLET,
            'startblock': START_BLOCK,
            'endblock':   END_BLOCK,
            'page':       page,
            'offset':     PAGE_SIZE,
            'sort':       'asc',
            'apikey':     API_KEY,
            **extra_params
        }
        try:
            resp = requests.get(BASE_URL, params=params, timeout=15)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f'  ❌ Request error on page {page}: {e}')
            break

        if data.get('status') != '1':
            msg = data.get('message', '')
            result = data.get('result', '')
            if 'No transactions found' in str(result) or result == []:
                print(f'  ✅ No more records at page {page}.')
            else:
                print(f'  ⚠️  API message: {msg} | Result: {result}')
            break

        records = data['result']
        all_records.extend(records)
        print(f'  Page {page}: {len(records):,} records fetched (total: {len(all_records):,})')

        if len(records) < PAGE_SIZE:
            print('  ✅ All pages fetched.')
            break

        page += 1
        time.sleep(SLEEP_BETWEEN)

    return all_records


def save_and_preview(records, name):
    """Saves records to CSV and shows a preview."""
    if not records:
        print(f'  ⚠️  No data for {name} — skipping.')
        return None

    df = pd.DataFrame(records)

    if 'timeStamp' in df.columns:
        df['datetime'] = pd.to_datetime(df['timeStamp'].astype(int), unit='s')

    if 'value' in df.columns:
        df['value_eth'] = df['value'].astype(float) / 1e18

    if 'gasUsed' in df.columns and 'gasPrice' in df.columns:
        df['gas_fee_eth'] = (df['gasUsed'].astype(float) * df['gasPrice'].astype(float)) / 1e18

    path = f'{OUTPUT_DIR}/{name}.csv'
    df.to_csv(path, index=False)
    print(f'  💾 Saved {len(df):,} rows → {path}')
    display(df.head(3))
    return df

print('✅ Functions loaded')

## 1️⃣ Cell 5 — Wallet Transactions (Normal ETH)

In [ ]:
txns = fetch_all_pages('txlist')
df_txns = save_and_preview(txns, 'wallet_transactions')

## 2️⃣ Cell 6 — Internal Transactions (Contract Calls)

In [ ]:
internal = fetch_all_pages('txlistinternal')
df_internal = save_and_preview(internal, 'internal_transactions')

## 3️⃣ Cell 7 — ERC-20 Token Transfers

In [ ]:
erc20 = fetch_all_pages('tokentx')
df_erc20 = save_and_preview(erc20, 'erc20_transfers')

## 4️⃣ Cell 8 — NFT Transfers (ERC-721)

In [ ]:
erc721 = fetch_all_pages('tokennfttx')
df_nft = save_and_preview(erc721, 'nft_transfers')

## 5️⃣ Cell 9 — Smart Contract Event Logs

In [ ]:
CONTRACT_ADDRESS = WALLET
TOPIC0 = ''

print('\n📡 Fetching [contract event logs] ...')
log_params = {
    'chainid':   CHAIN_ID,
    'module':    'logs',
    'action':    'getLogs',
    'address':   CONTRACT_ADDRESS,
    'fromBlock': START_BLOCK,
    'toBlock':   END_BLOCK,
    'apikey':    API_KEY,
}
if TOPIC0:
    log_params['topic0'] = TOPIC0

try:
    resp = requests.get(BASE_URL, params=log_params, timeout=15)
    data = resp.json()
    result = data.get('result', [])
    # V2 returns a list of dicts for logs
    if isinstance(result, list) and len(result) > 0:
        df_logs = save_and_preview(result, 'contract_event_logs')
    else:
        print(f'  ⚠️  No logs found. Message: {data.get("message")}')
        df_logs = None
except Exception as e:
    print(f'  ❌ Logs fetch error: {e}')
    df_logs = None

## 📊 Cell 10 — Summary & Quick Charts

In [ ]:
import matplotlib.pyplot as plt

print('=' * 50)
print('✅ FETCH COMPLETE — Summary')
print('=' * 50)

datasets = [
    ('Wallet Transactions',   df_txns),
    ('Internal Transactions', df_internal),
    ('ERC-20 Transfers',      df_erc20),
    ('NFT Transfers',         df_nft),
]

for name, df in datasets:
    count = len(df) if df is not None else 0
    print(f'  {name:30s}: {count:,} rows')

print(f'\n📁 All CSVs saved to: {OUTPUT_DIR}')

labels = [n for n, d in datasets if d is not None and len(d) > 0]
counts = [len(d) for _, d in datasets if d is not None and len(d) > 0]

if counts:
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(labels, counts, color=['#3B82F6','#8B5CF6','#10B981','#F59E0B'])
    ax.bar_label(bars, fmt='%d', padding=3)
    ax.set_title('Records Fetched per Data Type', fontsize=13, fontweight='bold')
    ax.set_ylabel('Row Count')
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fetch_summary_chart.png', dpi=150)
    plt.show()
    print('📈 Chart saved to Drive.')
else:
    print('⚠️  No data fetched — check your API key and wallet address.')

## 📥 Cell 11 — Download CSVs to Your Computer

In [ ]:
from google.colab import files
import os

csv_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.csv')]

if csv_files:
    for fname in csv_files:
        print(f'⬇️  Downloading {fname} ...')
        files.download(f'{OUTPUT_DIR}/{fname}')
    print('\n✅ All downloads triggered!')
else:
    print('⚠️  No CSV files found to download. Make sure the fetch cells ran successfully.')